# Cotton Disease Detection — v3 (corrected protocol, original + de-duplicated splits)

This notebook addresses the reviewer's mandatory corrections. It does **only the GPU work** and saves complete, image-level records; every statistic, table and figure in the paper is computed afterwards, locally, by `scripts/extended_analysis.py`.

**Changes from v2**

1. **Training fix.** Fresh `EarlyStopping` / `ReduceLROnPlateau` instances in each phase. After phase 2, the phase-1 and phase-2 checkpoints are compared on validation loss and the better one is kept (*phase-aware checkpoint selection*). In v2, callback state leaked from phase 1 into phase 2 (Keras 3 does not reset `best`), so most models got one fine-tuning epoch.
2. **Two splits.** `original` = the publisher's split. `dedup` = the same split after removing cross-split near-duplicates (pHash Hamming distance ≤ 5). For every duplicate group that touches the test set, the test copy is kept and the train/val copies are removed, so **both splits share the same test images**, enabling a paired comparison.
3. **Image-level records.** Every prediction is saved with its file path and class probabilities; the full file manifest, all duplicate pairs, and a contact sheet for manual verification are written to `data/`.
4. **Explainability.** Grad-CAM with deletion/insertion-AUC on **all test images, every seed, both splits**. Perturbations are applied in raw RGB pixel space *before* each model's preprocessing (deletion → black, insertion → Gaussian-blurred image, following RISE), so the baseline is identical for all backbones.
5. **Latency.** Single-image CPU latency of graph-mode model calls (median of 50), re-measured for all seed-42 models in one session, plus a TensorFlow Lite benchmark (float32 and dynamic-range int8) on the final cell.

**Corrections after the first complete v3 run (2026-10-14).** Its results are archived and none are reused; all 30 runs are repeated.

6. **BatchNorm stays frozen in fine-tuning.** Keras 3 passes `fit()`'s `training=True` into nested models, overriding `base(inp, training=False)`. In the first run, the unfrozen BatchNorm layers therefore normalised with batch statistics and overwrote their moving averages during phase 2. Training loss jumped from ≈0.2 to ≈1.0 on the first fine-tuning epoch, and fine-tuning never won for EfficientNetB0. All BatchNorm layers are now non-trainable, which forces inference mode.
7. **Grad-CAM layer.** The target is now the backbone's final feature map, defined the same way for every backbone. The last `Conv2D` layer was used before; in DenseNet121 that layer carries only 32 of the 1,024 final channels.
8. **Head phase cap** raised from 12 to 20 epochs (in 11/30 runs the best epoch was the last one). **All 30 models are saved**, and CPU latency is timed in graph mode.
9. **One process per run.** Each training run executes in a fresh subprocess (`run_one.py`); a crashed or hung run is retried once and otherwise left for the resume. The first v3.1 attempt, in a single kernel, was killed by memory growth after about 9 h.

**Runtime:** 2 splits × 3 backbones × 5 seeds = 30 runs, ≈ 10–14 h on a Kaggle GPU, so expect one resume. Every run is check-pointed: if a session stops, attach this notebook's previous output as an input, set `RESUME_FROM_INPUT`, and run again — finished runs are skipped.

**Outputs** (`/kaggle/working`): `package_versions.txt`, `data/` (manifest, duplicate pairs, split assignments, per-class counts, contact sheet), `runs/<split>/<backbone>_seed<seed>.json` (one file per run), `models/` (all 30 models, ≈1.1 GB), `figures/gradcam_gallery_<split>.png`, `latency_benchmark.csv`, `tflite_benchmark.json`, `logs/` (one log per run), `cotton_v3_lib.py` and `run_one.py` (the exact code that ran).

## 0. Shared code

The next six cells only write two files, `cotton_v3_lib.py` and `run_one.py`; they do not train anything. **Edit the configuration (including `RESUME_FROM_INPUT`) in the second cell.** Each training run then executes in its own Python process (`run_one.py`), so the memory TensorFlow accumulates across model builds is released after every run. In the first v3.1 attempt, a single long-lived kernel was killed after about 9 hours.

In [ ]:
%%writefile /kaggle/working/cotton_v3_lib.py
"""Shared code for cotton_disease_v3.ipynb. Written by the notebook and imported by
both the notebook kernel and run_one.py (one fresh subprocess per training run)."""
import os, sys, json, time, gc, platform, shutil, hashlib, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV3Small, EfficientNetB0, DenseNet121
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mn_pre
from tensorflow.keras.applications.efficientnet import preprocess_input as ef_pre
from tensorflow.keras.applications.densenet import preprocess_input as dn_pre
from sklearn.metrics import accuracy_score, f1_score
import cv2
from PIL import Image
import imagehash
import psutil

SESSION_START = time.time()   # used by the time guard in the training loop
for _g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_g, True)
    except Exception as e:
        print("memory growth not set:", e)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
%%writefile -a /kaggle/working/cotton_v3_lib.py

# ---------------------------------------------------------------- config
DATA_ROOT_CANDIDATES = [
    "/kaggle/input/datasets/janmejaybhoi/cotton-disease-dataset/Cotton Disease",
    "/kaggle/input/cotton-disease-dataset/Cotton Disease",
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if os.path.isdir(p)), None)
if DATA_ROOT is None:
    raise FileNotFoundError("Cotton Disease dataset not found under:\n  " + "\n  ".join(DATA_ROOT_CANDIDATES)
                            + "\nAttach it via 'Add Data' and add its mount path (File Browser panel) above.")
print("DATA_ROOT:", DATA_ROOT)

CLASS_NAMES = ["diseased cotton leaf", "diseased cotton plant", "fresh cotton leaf", "fresh cotton plant"]
SPLIT_DIRS = {"train": "train", "val": "val", "test": "test"}
IMG_SIZE, BATCH = 224, 32
EPOCHS_HEAD, EPOCHS_FT = 20, 18   # head cap raised from 12: 11/30 heads were still improving at epoch 12
LR_HEAD, LR_FT = 1e-3, 1e-5
FREEZE_FRAC = 0.7
NUM_CLASSES = 4
SEEDS = [42, 1042, 2042, 3042, 4042]
SPLITS = ["original", "dedup"]
HAMMING_THRESHOLD = 5
XAI_STEPS = 20
INSERTION_BLUR_SIGMA = 10.0      # Gaussian blur (pixels) for the insertion canvas, as in RISE
GALLERY_SEED = 42
# Kaggle kills a session at 12 h and may then discard output. Stop starting new runs
# once this budget is used, so the version finishes cleanly and can be resumed.
SESSION_BUDGET_HOURS = 10.5
MAX_RUN_HOURS = 0.75         # pessimistic duration of one run, used by the guard

# Resume: set to the mounted path of this notebook's previous output, e.g.
# "/kaggle/input/datasets/<user>/<dataset-slug>" (check the File Browser panel).
RESUME_FROM_INPUT = None

OUT = "/kaggle/working"
DATA_OUT, RUNS_OUT, MODELS_OUT, FIG_OUT = (f"{OUT}/{d}" for d in ("data", "runs", "models", "figures"))
for d in (DATA_OUT, RUNS_OUT, MODELS_OUT, FIG_OUT, *(f"{RUNS_OUT}/{s}" for s in SPLITS)):
    os.makedirs(d, exist_ok=True)


def atomic_write_json(path, obj):
    tmp = f"{path}.tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f)
    os.replace(tmp, path)

In [ ]:
%%writefile -a /kaggle/working/cotton_v3_lib.py

BACKBONES = {"MobileNetV3Small": (MobileNetV3Small, mn_pre),
             "EfficientNetB0": (EfficientNetB0, ef_pre),
             "DenseNet121": (DenseNet121, dn_pre)}


_MANIFEST = None


def split_frame(version, split):
    # data/manifest.csv is written by the duplicate-audit cell of the notebook
    global _MANIFEST
    if _MANIFEST is None:
        _MANIFEST = pd.read_csv(f"{DATA_OUT}/manifest.csv")
    m = _MANIFEST[_MANIFEST.split == split]
    if version == "dedup":
        m = m[m.keep_dedup]
    return m.reset_index(drop=True)


def make_generators(version, preprocess_fn, seed):
    aug = ImageDataGenerator(preprocessing_function=preprocess_fn, rotation_range=25,
                             width_shift_range=0.15, height_shift_range=0.15, shear_range=0.15,
                             zoom_range=0.2, horizontal_flip=True, vertical_flip=False,
                             brightness_range=[0.8, 1.2], fill_mode="nearest")
    plain = ImageDataGenerator(preprocessing_function=preprocess_fn)
    kw = dict(x_col="filepath", y_col="cls", classes=CLASS_NAMES, class_mode="categorical",
              target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, validate_filenames=False)
    tr = aug.flow_from_dataframe(split_frame(version, "train"), shuffle=True, seed=seed, **kw)
    va = plain.flow_from_dataframe(split_frame(version, "val"), shuffle=False, **kw)
    te_df = split_frame(version, "test")
    te = plain.flow_from_dataframe(te_df, shuffle=False, **kw)
    return tr, va, te, te_df


def class_weights(gen):
    c = np.bincount(gen.classes, minlength=NUM_CLASSES)
    return {i: c.sum() / (NUM_CLASSES * n) for i, n in enumerate(c)}


def build_model(backbone_cls):
    base = backbone_cls(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return models.Model(inp, out), base


def fresh_callbacks():
    # NEW instances every phase: Keras 3 keeps EarlyStopping/ReduceLROnPlateau `best`
    # across fit() calls, which is what went wrong in v2.
    return [callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
            callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)]


def hist_dict(h):
    return {k: [float(v) for v in vals] for k, vals in h.history.items()}

In [ ]:
%%writefile -a /kaggle/working/cotton_v3_lib.py

# ----------------------------------------------- Grad-CAM + pixel-space XAI
def find_backbone(model):
    return next(l for l in model.layers if isinstance(l, tf.keras.Model))


def make_grad_model(model):
    """Grad-CAM target = the backbone's final feature map (the tensor fed to global
    pooling), defined the same way for every backbone. The last Conv2D layer is NOT
    comparable: in DenseNet121 it outputs only 32 of the 1,024 final channels."""
    return find_backbone(model)


def gradcam(model, x_pre, class_idx, grad_model=None):
    grad_model = grad_model or make_grad_model(model)
    with tf.GradientTape() as tape:
        conv_out = grad_model(tf.convert_to_tensor(x_pre[None]), training=False)
        tape.watch(conv_out)
        y = conv_out
        for l in model.layers[2:]:
            y = l(y, training=False)
        score = y[:, class_idx]
    g = tape.gradient(score, conv_out)
    w = tf.reduce_mean(g, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv_out[0] * w, axis=-1)).numpy()
    cam = cam / (cam.max() + 1e-8)
    return cv2.resize(cam, (IMG_SIZE, IMG_SIZE))


def load_raw(path):
    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
    return np.asarray(img, dtype=np.float32)        # 0..255 RGB, same resizing as Keras loader


def deletion_insertion(model, preprocess_fn, raw, cam, cls):
    order = np.argsort(-cam.ravel())
    n = order.size
    step = n // XAI_STEPS
    blurred = cv2.GaussianBlur(raw, (0, 0), INSERTION_BLUR_SIGMA)
    dele, ins = np.repeat(raw[None], XAI_STEPS + 1, 0), np.repeat(blurred[None], XAI_STEPS + 1, 0)
    flat_d, flat_i, flat_r = dele.reshape(XAI_STEPS + 1, n, 3), ins.reshape(XAI_STEPS + 1, n, 3), raw.reshape(n, 3)
    for k in range(XAI_STEPS + 1):
        idx = order[:k * step]
        flat_d[k, idx] = 0.0                        # deletion: salient pixels -> black
        flat_i[k, idx] = flat_r[idx]                # insertion: reveal salient pixels on blurred canvas
    batch = np.concatenate([dele, ins])
    p = model(preprocess_fn(batch.copy()), training=False).numpy()[:, cls]
    trapz = getattr(np, "trapezoid", None) or np.trapz
    return float(trapz(p[:XAI_STEPS + 1]) / XAI_STEPS), float(trapz(p[XAI_STEPS + 1:]) / XAI_STEPS)


def xai_all(model, preprocess_fn, te_df, y_prob):
    recs = []
    gm = make_grad_model(model)
    for i, r in te_df.iterrows():
        raw = load_raw(r.filepath)
        pred = int(np.argmax(y_prob[i]))
        cam = gradcam(model, preprocess_fn(raw.copy()), pred, gm)
        d, ins = deletion_insertion(model, preprocess_fn, raw, cam, pred)
        recs.append(dict(relpath=r.relpath, pred=pred, conf=float(y_prob[i][pred]),
                         deletion_auc=d, insertion_auc=ins))
    del gm
    return recs


def cpu_latency(model, reps=50, warmup=10):
    with tf.device("/CPU:0"):
        m = tf.keras.models.clone_model(model)
        m.set_weights(model.get_weights())
        x = tf.constant(np.random.rand(1, IMG_SIZE, IMG_SIZE, 3).astype(np.float32) * 255)
        # graph mode: eager calls add per-layer Python dispatch overhead that swamps compute
        f = tf.function(lambda t: m(t, training=False))
        for _ in range(warmup):
            f(x).numpy()
        t = []
        for _ in range(reps):
            t0 = time.perf_counter()
            f(x).numpy()
            t.append((time.perf_counter() - t0) * 1000)
    del m, f
    return dict(median_ms=float(np.median(t)), mean_ms=float(np.mean(t)),
                p25_ms=float(np.percentile(t, 25)), p75_ms=float(np.percentile(t, 75)))

In [ ]:
%%writefile -a /kaggle/working/cotton_v3_lib.py

def train_one(version, name, seed):
    backbone_cls, pre = BACKBONES[name]
    keras.utils.set_random_seed(seed)
    tr, va, te, te_df = make_generators(version, pre, seed)
    cw = class_weights(tr)
    model, base = build_model(backbone_cls)
    t0 = time.time()

    # phase 1: head only
    model.compile(optimizer=optimizers.Adam(LR_HEAD), loss="categorical_crossentropy", metrics=["accuracy"])
    h1 = model.fit(tr, validation_data=va, epochs=EPOCHS_HEAD, callbacks=fresh_callbacks(),
                   class_weight=cw, verbose=2)
    val1 = model.evaluate(va, verbose=0)
    p1_path = f"{MODELS_OUT}/_phase1_tmp.weights.h5"
    model.save_weights(p1_path)

    # phase 2: unfreeze top (1 - FREEZE_FRAC) of the backbone
    base.trainable = True
    for l in base.layers[:int(len(base.layers) * FREEZE_FRAC)]:
        l.trainable = False
    # Keep EVERY BatchNorm layer frozen. Keras 3 passes fit()'s training=True into nested
    # models, overriding base(inp, training=False) (keras/src/models/functional.py,
    # operation_fn), so a trainable BN layer would normalise with batch statistics and
    # overwrite its moving averages. A non-trainable BN layer always runs in inference mode.
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False
    model.compile(optimizer=optimizers.Adam(LR_FT), loss="categorical_crossentropy", metrics=["accuracy"])
    h2 = model.fit(tr, validation_data=va, epochs=EPOCHS_FT, callbacks=fresh_callbacks(),
                   class_weight=cw, verbose=2)
    val2 = model.evaluate(va, verbose=0)

    # phase-aware checkpoint selection on validation loss
    selected = 2
    if val1[0] < val2[0]:
        model.load_weights(p1_path)
        selected = 1
    os.remove(p1_path)
    train_time = time.time() - t0
    val_sel = model.evaluate(va, verbose=0)

    te.reset()
    y_prob = model.predict(te, verbose=0)
    y_true = te.classes
    y_pred = y_prob.argmax(1)
    print(f"[{version}/{name}/seed {seed}] selected phase {selected} "
          f"(val loss p1={val1[0]:.4f}, p2={val2[0]:.4f}); test acc={accuracy_score(y_true, y_pred):.4f}")

    xai = xai_all(model, pre, te_df, y_prob)
    lat = cpu_latency(model)
    wpath = f"{MODELS_OUT}/_size_tmp.weights.h5"
    model.save_weights(wpath)
    size_mb = os.path.getsize(wpath) / 2**20
    os.remove(wpath)
    # every model is kept (~1.1 GB in total), so XAI or an external test set can be
    # re-run later without retraining
    model.save(f"{MODELS_OUT}/{version}_{name}_seed{seed}.keras")

    rec = dict(version=version, backbone=name, seed=seed, class_names=CLASS_NAMES,
               n_train=int(tr.samples), n_val=int(va.samples), n_test=int(te.samples),
               class_weights={str(k): float(v) for k, v in cw.items()},
               test_relpaths=te_df.relpath.tolist(), y_true=[int(v) for v in y_true],
               y_pred=[int(v) for v in y_pred], y_prob=np.round(y_prob, 6).tolist(),
               test_accuracy=float(accuracy_score(y_true, y_pred)),
               test_macro_f1=float(f1_score(y_true, y_pred, average="macro")),
               val_loss_phase1=float(val1[0]), val_acc_phase1=float(val1[1]),
               val_loss_phase2=float(val2[0]), val_acc_phase2=float(val2[1]),
               val_loss_selected=float(val_sel[0]), val_acc_selected=float(val_sel[1]),
               selected_phase=selected, history_phase1=hist_dict(h1), history_phase2=hist_dict(h2),
               params=int(model.count_params()), weights_mb=float(size_mb), cpu_latency=lat,
               train_time_s=float(train_time), xai=xai,
               bn_frozen_in_finetune=True,
               xai_protocol=dict(steps=XAI_STEPS, gradcam_layer=find_backbone(model).layers[-1].name,
                                 deletion="black (0) in raw RGB",
                                 insertion=f"Gaussian blur sigma={INSERTION_BLUR_SIGMA} in raw RGB"))
    del model
    gc.collect()
    tf.keras.backend.clear_session()
    gc.collect()
    rss = psutil.Process().memory_info().rss / 2**30
    try:
        gpu = tf.config.experimental.get_memory_info("GPU:0")["current"] / 2**30
    except Exception:
        gpu = float("nan")
    print(f"    memory after run: RAM {rss:.2f} GB, GPU {gpu:.2f} GB", flush=True)
    return rec

In [ ]:
%%writefile /kaggle/working/run_one.py
"""Train and evaluate ONE run in a fresh process: python run_one.py <split> <backbone> <seed>."""
import sys
from cotton_v3_lib import *

v, b, s = sys.argv[1], sys.argv[2], int(sys.argv[3])
print(f"worker pid {os.getpid()}: {v}/{b}/seed {s}", flush=True)
rec = train_one(v, b, s)
atomic_write_json(f"{RUNS_OUT}/{v}/{b}_seed{s}.json", rec)
print("worker done", flush=True)

In [ ]:
# ------------------------------------------- kernel setup (not part of the library)
import sys
sys.path.insert(0, "/kaggle/working")
from cotton_v3_lib import *

if RESUME_FROM_INPUT:
    for sub in ("runs", "models"):
        src = os.path.join(RESUME_FROM_INPUT, sub)
        if os.path.isdir(src):
            shutil.copytree(src, f"{OUT}/{sub}", dirs_exist_ok=True)
            print(f"Carried forward {sub}/ from {src}")
        else:
            print(f"WARNING: {src} not found -- check RESUME_FROM_INPUT")

import sklearn, scipy
versions = {"python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
            "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
            "scipy": scipy.__version__, "opencv": cv2.__version__, "imagehash": imagehash.__version__,
            "gpu": str(tf.config.list_physical_devices("GPU")), "platform": platform.platform()}
with open(f"{OUT}/package_versions.txt", "w") as f:
    f.write("\n".join(f"{k}: {v}" for k, v in versions.items()) + "\n")
print(versions)

## 1. File manifest, duplicate audit and de-duplicated split

In [ ]:
rows = []
for split, sub in SPLIT_DIRS.items():
    for cls in CLASS_NAMES:
        d = os.path.join(DATA_ROOT, sub, cls)
        for fn in sorted(os.listdir(d)):
            p = os.path.join(d, fn)
            if not os.path.isfile(p):
                continue
            try:
                with Image.open(p) as im:
                    im = im.convert("RGB")
                    ph = str(imagehash.phash(im))
                    w, h = im.size
            except Exception as e:
                print("skipped unreadable", p, e)
                continue
            with open(p, "rb") as fb:
                md5 = hashlib.md5(fb.read()).hexdigest()
            rows.append(dict(split=split, cls=cls, relpath=f"{sub}/{cls}/{fn}", filepath=p,
                             phash=ph, md5=md5, width=w, height=h))
manifest = pd.DataFrame(rows)
print(manifest.groupby(["split", "cls"]).size().unstack())

H = np.array([imagehash.hex_to_hash(h).hash.flatten() for h in manifest.phash], dtype=bool)
N = len(H)
pairs = []
for i in range(N):
    d = np.count_nonzero(H[i + 1:] != H[i], axis=1)
    for j in np.where(d <= HAMMING_THRESHOLD)[0] + i + 1:
        pairs.append(dict(i=i, j=int(j), hamming=int(d[j - i - 1])))
pairs = pd.DataFrame(pairs)
for c in ("split", "cls", "relpath", "md5"):
    pairs[f"{c}_a"] = manifest[c].values[pairs.i]
    pairs[f"{c}_b"] = manifest[c].values[pairs.j]
pairs["identical_bytes"] = pairs.md5_a == pairs.md5_b
pairs["cross_split"] = pairs.split_a != pairs.split_b
pairs["cross_class"] = pairs.cls_a != pairs.cls_b
print(f"{len(pairs)} near-duplicate pairs (Hamming <= {HAMMING_THRESHOLD}); "
      f"{pairs.cross_split.sum()} cross-split, {pairs.cross_class.sum()} cross-class (label conflict)")
print(pairs.groupby(["split_a", "split_b"]).size())

# connected components of the near-duplicate graph
parent = list(range(N))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for a, b in zip(pairs.i, pairs.j):
    parent[find(a)] = find(b)
manifest["group"] = [find(i) for i in range(N)]

# de-duplicated split: per duplicate group keep copies only in the highest-priority
# split present (test > val > train), and keep a single copy inside the test set
PRIORITY = {"test": 0, "val": 1, "train": 2}
keep = np.ones(N, dtype=bool)
reason = [""] * N
for g, idx in manifest.groupby("group").groups.items():
    idx = list(idx)
    if len(idx) == 1:
        continue
    best = min(PRIORITY[manifest.split[i]] for i in idx)
    test_kept = False
    for i in sorted(idx, key=lambda k: manifest.relpath[k]):
        s = manifest.split[i]
        if PRIORITY[s] != best:
            keep[i], reason[i] = False, f"duplicate of an image in {['test', 'val', 'train'][best]}"
        elif s == "test":
            if test_kept:
                keep[i], reason[i] = False, "within-test duplicate"
            test_kept = True
manifest["keep_dedup"] = keep
manifest["dedup_removal_reason"] = reason

counts = {"original": manifest.groupby(["split", "cls"]).size().unstack(fill_value=0),
          "dedup": manifest[manifest.keep_dedup].groupby(["split", "cls"]).size().unstack(fill_value=0)}
for k, v in counts.items():
    print(f"\n{k} split, images per class:\n{v}")
manifest.to_csv(f"{DATA_OUT}/manifest.csv", index=False)
pairs.to_csv(f"{DATA_OUT}/duplicate_pairs.csv", index=False)
pd.concat(counts, names=["version"]).to_csv(f"{DATA_OUT}/class_counts.csv")
print("\nRemoved for dedup split:\n", manifest[~manifest.keep_dedup].groupby(["split", "dedup_removal_reason"]).size())

In [ ]:
# contact sheet of every flagged pair, for manual verification by the authors
flag = pairs[pairs.cross_split | ((pairs.split_a == "test") & (pairs.split_b == "test"))]
flag = flag.sort_values(["split_a", "split_b", "hamming"]).head(120).reset_index(drop=True)
n = len(flag)
if n:
    cols = 4
    rws = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rws, cols * 2, figsize=(cols * 3.2, rws * 1.75))
    axes = np.atleast_2d(axes)
    for ax in axes.flat:
        ax.axis("off")
    for k, r in flag.iterrows():
        rr, cc = divmod(k, cols)
        for side, col in ((0, "a"), (1, "b")):
            ax = axes[rr, cc * 2 + side]
            img = Image.open(os.path.join(DATA_ROOT, r[f"relpath_{col}"])).convert("RGB").resize((96, 96))
            ax.imshow(img)
            ax.set_title(f"#{k} {r[f'split_{col}']}/{r[f'cls_{col}'][:12]}\nd={r.hamming}", fontsize=5)
    plt.tight_layout()
    plt.savefig(f"{DATA_OUT}/duplicate_pairs_contact_sheet.png", dpi=150)
    plt.show()

## 2. Training: one subprocess per run

In [ ]:
LOG_OUT = f"{OUT}/logs"
os.makedirs(LOG_OUT, exist_ok=True)
RUN_TIMEOUT_HOURS = 1.5     # a worker still running after this is killed (the first v3 attempt hung for 6 h)


def run_in_subprocess(v, b, s, attempt):
    """Run run_one.py in a fresh process, streaming its log into this notebook. Returns the exit code."""
    log = f"{LOG_OUT}/{v}_{b}_seed{s}_try{attempt}.log"
    env = dict(os.environ, TF_CPP_MIN_LOG_LEVEL="2", PYTHONUNBUFFERED="1")
    with open(log, "w") as fh:
        proc = subprocess.Popen([sys.executable, f"{OUT}/run_one.py", v, b, str(s)],
                                stdout=fh, stderr=subprocess.STDOUT, env=env, cwd=OUT)
    pos, t0 = 0, time.time()
    while True:
        rc = proc.poll()
        with open(log, errors="replace") as fh:
            fh.seek(pos)
            new = fh.read()
            pos = fh.tell()
        if new:
            print(new, end="", flush=True)
        if rc is not None:
            return rc
        if time.time() - t0 > RUN_TIMEOUT_HOURS * 3600:
            proc.kill()
            proc.wait()
            print(f"WORKER TIMEOUT after {RUN_TIMEOUT_HOURS} h -- killed", flush=True)
            return -9
        time.sleep(15)


jobs = [(v, b, s) for v in SPLITS for b in BACKBONES for s in SEEDS]
t_start, done_now = time.time(), 0
for k, (v, b, s) in enumerate(jobs, 1):
    path = f"{RUNS_OUT}/{v}/{b}_seed{s}.json"
    if os.path.exists(path):
        print(f"({k}/{len(jobs)}) {v}/{b}/seed {s}: already done -- skipped")
        continue
    used_h = (time.time() - SESSION_START) / 3600
    if used_h + MAX_RUN_HOURS > SESSION_BUDGET_HOURS:
        print(f"TIME GUARD: {used_h:.1f} h used -- stopping before run {k} so this version ends "
              f"cleanly. Resume: attach this version's output and set RESUME_FROM_INPUT.", flush=True)
        break
    print(f"({k}/{len(jobs)}) starting {v}/{b}/seed {s} at {used_h:.2f} h", flush=True)
    for attempt in (1, 2):       # one retry in a fresh process if the worker crashes or hangs
        rc = run_in_subprocess(v, b, s, attempt)
        if rc == 0 and os.path.exists(path):
            break
        print(f"({k}/{len(jobs)}) worker FAILED (exit code {rc}, attempt {attempt})", flush=True)
    if not os.path.exists(path):
        print(f"({k}/{len(jobs)}) {v}/{b}/seed {s} skipped after 2 failures -- it will be retried on resume")
        continue
    done_now += 1
    print(f"    notebook kernel RAM {psutil.Process().memory_info().rss / 2**30:.2f} GB", flush=True)
    el = time.time() - t_start
    left = sum(not os.path.exists(f"{RUNS_OUT}/{a}/{c}_seed{d}.json") for a, c, d in jobs)
    print(f"({k}/{len(jobs)}) saved. elapsed {el/60:.1f} min, ~{el/done_now*left/60:.0f} min remaining")

ALL_DONE = all(os.path.exists(f"{RUNS_OUT}/{v}/{b}_seed{s}.json") for v, b, s in jobs)
print("ALL RUNS COMPLETE" if ALL_DONE else "NOT ALL RUNS COMPLETE -- resume in a new version")
summary = []
for v, b, s in [j for j in jobs if os.path.exists(f"{RUNS_OUT}/{j[0]}/{j[1]}_seed{j[2]}.json")]:
    r = json.load(open(f"{RUNS_OUT}/{v}/{b}_seed{s}.json"))
    summary.append(dict(split=v, backbone=b, seed=s, test_acc=r["test_accuracy"],
                        macro_f1=r["test_macro_f1"], selected_phase=r["selected_phase"]))
summary = pd.DataFrame(summary)
print(summary.groupby(["split", "backbone"])[["test_acc", "macro_f1"]].agg(["mean", "std"]).round(4))
print(summary.groupby(["split", "backbone"]).selected_phase.value_counts().unstack(fill_value=0))

## 3. Grad-CAM gallery (seed 42, one test image per class)

In [ ]:
if not ALL_DONE:
    print("Gallery skipped: not all runs are complete yet.")
else:
    test_df = split_frame("original", "test")
    gallery = [test_df[test_df.cls == c].sample(1, random_state=0).relpath.iloc[0] for c in CLASS_NAMES]
    for v in SPLITS:
        fig, axes = plt.subplots(len(gallery), 1 + len(BACKBONES), figsize=(3 * (1 + len(BACKBONES)), 3 * len(gallery)))
        for r_, rel in enumerate(gallery):
            raw = load_raw(os.path.join(DATA_ROOT, rel))
            axes[r_, 0].imshow(raw.astype(np.uint8))
            axes[r_, 0].set_title(f"true: {rel.split('/')[1]}", fontsize=8)
            axes[r_, 0].axis("off")
            for c_, (b, (_, pre)) in enumerate(BACKBONES.items(), 1):
                m = tf.keras.models.load_model(f"{MODELS_OUT}/{v}_{b}_seed{GALLERY_SEED}.keras")
                prob = m(pre(raw[None].copy()), training=False).numpy()[0]
                cam = gradcam(m, pre(raw.copy()), int(prob.argmax()))
                axes[r_, c_].imshow(raw.astype(np.uint8))
                axes[r_, c_].imshow(cam, cmap="jet", alpha=0.45)
                axes[r_, c_].set_title(f"{b}\npred: {CLASS_NAMES[int(prob.argmax())]}", fontsize=8)
                axes[r_, c_].axis("off")
                del m
        plt.tight_layout()
        plt.savefig(f"{FIG_OUT}/gradcam_gallery_{v}.png", dpi=200)
        plt.show()
    json.dump(gallery, open(f"{FIG_OUT}/gradcam_gallery_images.json", "w"))

## 4. Latency benchmark (all models, one session)

Per-run latencies come from whichever Kaggle machine ran that run, and resumed versions run on different machines. For a fair comparison, all seed-42 models are timed here in the same session, interleaved over three rounds. **The paper reports these numbers.**

In [ ]:
if not ALL_DONE:
    print("Latency benchmark skipped: not all runs are complete yet.")
else:
    lat_rows = []
    keys = [(v, b) for v in SPLITS for b in BACKBONES]
    loaded = {k: tf.keras.models.load_model(f"{MODELS_OUT}/{k[0]}_{k[1]}_seed{GALLERY_SEED}.keras") for k in keys}
    for rnd in range(3):
        for k in keys:
            r = cpu_latency(loaded[k], reps=50, warmup=10)
            lat_rows.append(dict(round=rnd, version=k[0], backbone=k[1], **r))
            print(rnd, k, f"{r['median_ms']:.1f} ms", flush=True)
    del loaded
    gc.collect()
    lat_df = pd.DataFrame(lat_rows)
    lat_df.to_csv(f"{OUT}/latency_benchmark.csv", index=False)
    print(lat_df.groupby(["version", "backbone"]).median_ms.agg(["median", "min", "max"]).round(1))
    print("CPU:", platform.processor(), "| logical cores:", os.cpu_count())

## 5. Optional: TensorFlow Lite benchmark (de-duplicated split, seed 42)

Gives on-device-style evidence (model size and single-thread interpreter latency, float32 and dynamic-range int8, plus int8 test accuracy). It still runs on the Kaggle CPU, not a phone. Any failure here is caught and does not affect the results above.

In [ ]:
if not ALL_DONE:
    print("TFLite benchmark skipped: not all runs are complete yet.")
else:
    tfl = {}
    test_df = split_frame("dedup", "test")
    for b, (_, pre) in BACKBONES.items():
        try:
            m = tf.keras.models.load_model(f"{MODELS_OUT}/dedup_{b}_seed{GALLERY_SEED}.keras")
            sm = f"{MODELS_OUT}/_sm_{b}"
            m.export(sm)
            res = {}
            for mode in ("float32", "int8_dynamic"):
                conv = tf.lite.TFLiteConverter.from_saved_model(sm)
                if mode == "int8_dynamic":
                    conv.optimizations = [tf.lite.Optimize.DEFAULT]
                blob = conv.convert()
                interp = tf.lite.Interpreter(model_content=blob, num_threads=1)
                inp = interp.get_input_details()[0]
                interp.resize_tensor_input(inp["index"], [1, IMG_SIZE, IMG_SIZE, 3])
                interp.allocate_tensors()
                inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
                correct, t = 0, []
                for i, r in test_df.iterrows():
                    x = pre(load_raw(r.filepath)[None].copy()).astype(np.float32)
                    interp.set_tensor(inp["index"], x)
                    t0 = time.perf_counter()
                    interp.invoke()
                    t.append((time.perf_counter() - t0) * 1000)
                    correct += int(np.argmax(interp.get_tensor(out["index"])[0]) == CLASS_NAMES.index(r.cls))
                res[mode] = dict(size_mb=len(blob) / 2**20, median_latency_ms=float(np.median(t[5:])),
                                 test_accuracy=correct / len(test_df))
            tfl[b] = res
            shutil.rmtree(sm, ignore_errors=True)
            print(b, res)
        except Exception as e:
            tfl[b] = {"error": repr(e)}
            print(b, "TFLite benchmark failed:", e)
    json.dump(tfl, open(f"{OUT}/tflite_benchmark.json", "w"), indent=2)

In [ ]:
print("Runs completed:", sum(len(os.listdir(f"{RUNS_OUT}/{s}")) for s in SPLITS), "of", len(jobs))
print("Before ending the session: download the whole /kaggle/working folder (Output tab -> Download All).")